<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

<h1 style="text-align: center;">03 Mobile Systems</h1>
<hr>


## **7. Wi-Fi Simulation with Jemula802**

<div class="alert alert-light">

  **Name:** (add your name here)
  
  **Matriculation Number:** (add your number here)
  
</div>




### 7.1 Introduction

Jemula802 is a Java-based simulation framework for wireless networks, specifically designed around the IEEE 802.11 (Wi-Fi) MAC protocol. So in short, Jemula802 is a lightweight, Java simulator for IEEE 802.11 MAC protocols, mainly used in academia for prototyping and teaching.

* Name origin: **Jemula802** comes from **J**ava-based **E**xtensible **M**odular **ULA**tor for 802.11.
* Focus: It concentrates on simulating Medium Access Control (MAC) behavior of IEEE 802.11, not the entire protocol stack (so it doesn’t do full PHY-layer channel modeling like ns-2/ns-3).
* Modularity: The code is structured to make it easy to extend and test different MAC mechanisms (e.g., variations of CSMA/CA, backoff algorithms, QoS enhancements).
* Use cases:
  * Research into system protocol performance (throughput, delay, fairness).
  * Educational purposes (teaching how IEEE 802.11 medium access works).
  * Trying out new scheduling or contention algorithms without the overhead of large simulators.

Jemula802 is smaller, easier to understand, but less realistic at the physical layer compared to alternative and more widely used simulators for Wi-Fi, for example ns-3 (C++) and OMNeT++ (C++ with modules like INET).

Now let us clone and compile the related repositories.

In [ ]:
classpath_jemula = "./:jemula802/bin:jemula/bin:jemula/lib/xstream-1.3.1.jar:jemula/lib/jfreechart-1.5.3.jar:jemula/lib/jcommon-1.0.23.jar"
classpath_jemula802 = "./:jemula802/bin:jemula/bin:jemula/lib/xstream-1.3.1.jar:jemula/lib/jfreechart-1.5.3.jar:jemula/lib/jcommon-1.0.23.jar:jemula/lib/commons-io-2.14.0.jar"

import platform

if platform.system() == "Windows":
    classpath_jemula   = classpath_jemula.replace(":", ";")
    classpath_jemula802 = classpath_jemula802.replace(":", ";")


In [ ]:
%%bash
set -e
if [ -d "jemula802/.git" ]; then
  echo "Updating existing repo jemula802 ..."
  git -C jemula802 pull --ff-only
else
  echo "Cloning repo jemula802 ..."
  git clone https://bitbucket.org/lfield/jemula802.git
fi
set -e
if [ -d "jemula/.git" ]; then
  echo "Updating existing repo jemula ..."
  git -C jemula pull --ff-only
else
  echo "Cloning repo jemula ..."
  git clone https://bitbucket.org/lfield/jemula.git
fi


### 7.2 Compile the Java Code

In [ ]:
!javac -g:none -classpath $classpath_jemula -d "jemula/bin" jemula/src/plot/*.java jemula/src/kernel/*.java jemula/src/statistics/*.java
!echo Compilation of jemula done.

!javac -g:none  -classpath $classpath_jemula802 -d "jemula802/bin" jemula802/src/emulator/*.java jemula802/src/layer0_medium/*.java jemula802/src/layer1_802Phy/*.java \
 jemula802/src/layer1_80211Phy/*.java jemula802/src/layer2_802Mac/*.java jemula802/src/layer2_80211Mac/*.java \
 jemula802/src/layer2_802Algorithms/*.java jemula802/src/layer3_network/*.java jemula802/src/layer5_application/*.java \
 jemula802/src/layer4_transport/*.java jemula802/src/station/*.java jemula802/src/visualization/*.java \
 jemula802/src/util/*.java jemula802/src/gui/*.java
!echo Compilation of jemula802 done.

import os

ROOT_DIR = os.getcwd()
!echo Root directory is $ROOT_DIR

### 7.3 Run your first Example "Jemula802GettingStarted.xml"
Note the name of the XML Scenario file in the code below. This file, "[Jemula802GettingStarted.xml](jemula802/scenarios/examples/Jemula802GettingStarted.xml)", defines what will be simulated.

In [ ]:
EXAMPLE_SCENARIO="jemula802/scenarios/examples/Jemula802GettingStarted.xml"
!echo Running example scenario $EXAMPLE_SCENARIO
!java -Xmx500m -classpath $classpath_jemula802 emulator.JE802Starter $EXAMPLE_SCENARIO

### 7.4 Check the Results
#### 7.4.1 Where is the Data?

A new folder "results_GettingStarted" is created by the simulation. The simulation results and a copy of the original XML scenario file are stored in this folder.

#### 7.4.2 Google Earth Visualization
Locate it, and find the kml file to illustrate the scenario with Google Earth Pro ([Desktop version, not the Web version](https://www.google.com/earth/about/versions/#earth-pro)). This free version is available on Microsoft Windows, macOS, and GNU/Linux. When opening the kml file with Google Earth Pro, a scenario with two radio devices in front of the ETH  will show up.

#### 7.4.3 Statistical Data Analysis
Use Python to visualize the simulation results:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os

if os.path.isdir("results_GettingStarted"):
    os.chdir("results_GettingStarted")

In [ ]:
def plot_thrp_offer_delay(result_total_offer, result_total_thrp, result_total_delay):
    fig, ax = plt.subplots(figsize=(8, 4))
    theLineWidth = 2
    ax.cla()
    ax.grid(color="#AAAAAA", linewidth=.5, linestyle='-', alpha=1)
    ax.axis([None, None, None, None])
    ax.set_facecolor((.2, 0.2, 0.3))
    plt.xlim(0, np.max(result_total_offer.iloc[:, 0]) / 1000 * 1.05)  # Set x-axis limits
    plt.xlabel('Simulated Time [s]')
    plt.ylabel('Simulation Results, Mb/s & ms')
    plt.grid(True)  # Turn on the grid
    plt.minorticks_on()  # Enable minor ticks

    plt.plot(result_total_offer.iloc[:, 0] / 1000, result_total_offer.iloc[:, 6], '-', color='g', linewidth=4, label='Offer [Mb/s]')
    plt.plot(result_total_thrp.iloc[:, 0] / 1000, result_total_thrp.iloc[:, 6], '-', color='w', linewidth=1, label='Throughput [Mb/s]')
    plt.plot(result_total_delay.iloc[:, 0] / 1000, result_total_delay.iloc[:, 3], '-', color='r', linewidth=1, label='Avg. Delay [ms]')
    #plt.plot(result_total_delay.iloc[:, 0] / 1000, result_total_delay.iloc[:, 5], '-', color='r', linewidth=1)
    plt.legend(fancybox=True, framealpha=1, facecolor=(.6, 0.61, 0.8))
    plt.tight_layout()

def plot_delay_histogram(aBinSize_ms_string, aHistogram_string):
    fig, ax_distr = plt.subplots(figsize=(8, 4))
    theLineWidth = 2
    ax_distr.cla()
    ax_distr.grid(color="#AAAAAA", linewidth=.5, linestyle='-', alpha=1)
    ax_distr.axis([None, None, None, None])
    ax_distr.set_facecolor((.2, 0.2, 0.3))
    theHistogram = [float(x) for x in aHistogram_string.split()]
    theHistogram = [x/sum(theHistogram) for x in theHistogram]
    theNumOfBins = len(theHistogram)
    theBinSize_ms = float(aBinSize_ms_string)
    theX = [i * theBinSize_ms + theBinSize_ms/2 for i in range(theNumOfBins)] 
    theCumSum = 1-np.cumsum(theHistogram)
    theCumSum = np.insert(theCumSum, 0, 1)
    plt.step([0] + theX, [0] + theHistogram, '-', color='r', linewidth=2, label='PDF')
    plt.step([0] + theX, theCumSum, '-', color='m', linewidth=2, label='CDF')
    plt.grid(True)  # Turn on the grid
    plt.minorticks_on()  # Enable minor ticks
    plt.xlim(0, theNumOfBins * theBinSize_ms * 1.05)  # Set x-axis limits
    plt.xlabel('x [ms]')
    plt.ylabel('PDF: P(delay=x) & CCDF: P(delay>=x)')
    plt.legend(fancybox=True, framealpha=1, facecolor=(.6, 0.61, 0.8))
    plt.tight_layout()

Plot the results for the total system, all traffic:

In [ ]:
if os.path.isdir("results_GettingStarted"):
    os.chdir("results_GettingStarted")
result_total_offer = pd.read_csv('total_offer.csv', skiprows=2)
result_total_thrp = pd.read_csv('total_thrp.csv', skiprows=2)
result_total_delay = pd.read_csv('total_delay.csv', skiprows=2)
plot_thrp_offer_delay(result_total_offer, result_total_thrp, result_total_delay)
plot_delay_histogram(result_total_delay.iloc[-1,7],result_total_delay.iloc[-1,8])


Plot the results for the Access Categories (ACs), per priority:

* Access Category 1:

In [ ]:
if os.path.isdir("results_GettingStarted"):
    os.chdir("results_GettingStarted")
result_AC1_offer = pd.read_csv('offer_AC1.csv', skiprows=2)
result_AC1_thrp = pd.read_csv('thrp_AC1.csv', skiprows=2)
result_AC1_delay = pd.read_csv('delay_AC1.csv', skiprows=2)
plot_thrp_offer_delay(result_AC1_offer, result_AC1_thrp, result_AC1_delay)
plot_delay_histogram(result_AC1_delay.iloc[-1,7],result_AC1_delay.iloc[-1,8])

* Access Category 2:

In [ ]:

if os.path.isdir("results_Jemula802GettingStarted"):
    os.chdir("results_Jemula802GettingStarted")
result_AC2_offer = pd.read_csv('offer_AC2.csv', skiprows=2)
result_AC2_thrp = pd.read_csv('thrp_AC2.csv', skiprows=2)
result_AC2_delay = pd.read_csv('delay_AC2.csv', skiprows=2)
plot_thrp_offer_delay(result_AC2_offer, result_AC2_thrp, result_AC2_delay)
plot_delay_histogram(result_AC2_delay.iloc[-1,7],result_AC2_delay.iloc[-1,8])

In IEEE 802.11 with Quality of Service (QoS), an access category (AC) is a priority level that determines how traffic is treated when competing for access to the wireless medium. The standard defines four access categories: voice (AC\_VO), video (AC\_VI), best effort (AC\_BE), and background (AC\_BK). Each category is mapped from user priorities and is assigned its own Enhanced Distributed Channel Access (EDCA) parameters, including minimum and maximum contention window sizes, arbitration inter-frame space number (AIFSN), and transmission opportunity (TXOP) limits. By giving voice the smallest contention window and shortest inter-frame spacing, the system ensures that delay-sensitive packets like VoIP frames have the highest chance of accessing the channel quickly, while less time-critical traffic like file transfers in the background waits longer. This mechanism allows Wi-Fi to provide differentiated service levels for different types of traffic without requiring centralized scheduling, balancing fairness with prioritization.

----
### 7.5 Hands-On Exercises

<div class="alert alert-block alert-info">Fill in your responses and save this notebook. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>



**How does the throughput differ between overall throughput and the individual throughputs of AC1 and AC2?** 

Answer:

(add your answer here)

**How does the delay differ between overall delay and the individual delays of AC1 and AC2?** 

Answer:

(add your answer here)

**Which AC operates with higher priority?** 

Answer:

(add your answer here)

**In the XML scenario file "[Jemula802GettingStarted.xml](jemula802/scenarios/examples/Jemula802GettingStarted.xml)", there is a parameter called `seed` (see line 5 of the file). What would change if this value changed from 11 to 10?** 

Answer:

(add your answer here)

**In the introduction of this class, we talk about the concept of radio regulation. Explain briefly the difference between radio regulation and standardization. Also, explain why not everything in a radio system is standardized, and some parts like algorithms remain trade secrets and proprietary to individual parties.**

Answer:

**OPTIONAL: Did you read through and understand all the tasks in this Notebook? Are there any shortcomings, missing parts, or sections that should be improved?**

 (please help the team by telling us what part was not understood, and we will work on it)